# Prep Data

In [ ]:
from datasets import load_dataset

ds = load_dataset("opencampus/chest-xray-pneumonia-3class-balanced")

In [ ]:
print(ds)
split_name = "train" if "train" in ds else list(ds.keys())[0]
print("Split dipakai:", split_name)
print("Contoh 1 sample:", ds[split_name][0].keys())
print("Features:", ds[split_name].features)

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# ==========================================
# BAGIAN A: AMBIL GAMBAR & LABEL DARI HUGGING FACE `ds`
# ==========================================
IMG_SIZE = (224, 224)
split_name = "train" if "train" in ds else list(ds.keys())[0]
hf_split = ds[split_name]

# Deteksi nama kolom gambar & label secara aman
img_col = "image" if "image" in hf_split.column_names else "img"
label_col = "label" if "label" in hf_split.column_names else "labels"

X_list = []
y_list = []

for item in hf_split:
  img_pil = item[img_col]
  label = item[label_col]

  # Pastikan gambar jadi RGB (3 channel) lalu resize ke (224, 224)
  if hasattr(img_pil, "convert"):
    img_rgb = np.array(img_pil.convert("RGB"))
  else:
    img_rgb = np.array(img_pil)
    if img_rgb.ndim == 2:
      img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_GRAY2RGB)
    elif img_rgb.shape[-1] == 4:
      img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_RGBA2RGB)

  img_resized = cv2.resize(img_rgb, IMG_SIZE)
  X_list.append(img_resized)
  y_list.append(label)

X = np.array(X_list, dtype=np.uint8)
y = np.array(y_list, dtype=np.int64)

print(f"Total gambar: {len(X)} | Shape X: {X.shape} | Shape y: {y.shape}")

# ==========================================
# BAGIAN B: SPLIT, PREPROCESS, TRAIN & SAVE
# ==========================================
# 1. Split data train & validation (mencegah data leakage)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Normalisasi/preprocessing konsisten untuk MobileNetV2 (skala -1 s.d. 1)
X_train_prep = preprocess_input(X_train.astype(np.float32))
X_val_prep = preprocess_input(X_val.astype(np.float32))

num_classes = len(np.unique(y))
print("Jumlah kelas:", num_classes)

# 3. Bangun model CNN dengan Transfer Learning (MobileNetV2)
base_model = MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet"
)
base_model.trainable = False  # bekukan dulu bobot awal agar latihan awal cepat & stabil

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# 4. Latih model (mulai dari 5 epoch dulu untuk cek alur)
history = model.fit(
    X_train_prep,
    y_train,
    validation_data=(X_val_prep, y_val),
    epochs=5,
    batch_size=32,
)

# 5. Simpan sebagai file .h5 di /kaggle/working/
model.save("/kaggle/working/pneumonia_cnn_model.h5")
print("Berhasil menyimpan /kaggle/working/pneumonia_cnn_model.h5")

In [ ]:
import numpy as np
from PIL import Image, ImageOps
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


def classify(image, model, class_names):
  """Melakukan preprocessing gambar (224x224 RGB, skala -1 s.d. 1) dan prediksi kelas."""
  # Pastikan format RGB (3 channel) dan ukuran (224, 224)
  image = image.convert("RGB")
  image = ImageOps.fit(image, (224, 224), Image.Resampling.LANCZOS)

  # Ubah ke numpy array & terapkan preprocessing MobileNetV2
  image_array = np.asarray(image, dtype=np.float32)
  normalized_image_array = preprocess_input(image_array)

  # Bentuk batch (1, 224, 224, 3)
  data = np.expand_dims(normalized_image_array, axis=0)

  # Prediksi & ambil kelas dengan probabilitas tertinggi
  prediction = model.predict(data, verbose=0)
  index = int(np.argmax(prediction[0]))
  class_name = class_names[index]
  confidence_score = float(prediction[0][index])

  return class_name, confidence_score

In [ ]:
import tensorflow as tf
from PIL import Image

# 1. Load model dari /kaggle/working/
loaded_model = tf.keras.models.load_model("/kaggle/working/pneumonia_cnn_model.h5")

# 2. Ambil daftar nama kelas sesuai urutan dataset Hugging Face
label_names = hf_split.features[label_col].names
print("Daftar kelas:", label_names)

# 3. Ambil 1 contoh gambar dari X_val dan konversi ke PIL Image
sample_idx = 0
pil_img = Image.fromarray(X_val[sample_idx])

# 4. Jalankan fungsi classify
pred_class, conf_score = classify(pil_img, loaded_model, label_names)
true_class = label_names[y_val[sample_idx]]

print(f"Label Asli   : {true_class}")
print(f"Hasil Prediksi: {pred_class} ({conf_score * 100:.2f}%)")

probs = loaded_model.predict(
    np.expand_dims(
        preprocess_input(
            np.asarray(
                ImageOps.fit(
                    Image.fromarray(X_val[sample_idx]).convert("RGB"),
                    (224, 224),
                    Image.Resampling.LANCZOS,
                ),
                dtype=np.float32,
            )
        ),
        axis=0,
    ),
    verbose=0,
)[0]

for nama, skor in zip(label_names, probs):
  print(f"{nama:20s}: {skor * 100:.2f}%")